# nn-module-subclass composite — cx3: Patchify subclass: nn.Module wrapping an einops Rearrange + Linear head

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `nn-module-subclass`, `rearrange-as-sequential-layer`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "nn-module-subclass"
DD_ATOM_IDS = ["nn-module-subclass", "rearrange-as-sequential-layer"]
DD_SUBTOPICS = ["PyTorch: nn.Module subclassing", "Einops: Rearrange as nn.Sequential layer"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

An `nn.Module` subclass can hold an `einops.layers.torch.Rearrange` as a regular child module — just assign it in `__init__` and it gets registered (visible in `.children()`, moves with `.to(device)`). Two atoms compose:

1. **nn-module-subclass** — define `class Patchify(nn.Module)`, call `super().__init__()`, register children by attribute assignment, implement `forward`.
2. **rearrange-as-sequential-layer** — instantiate `Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=patch, p2=patch)` and store it on `self`. This is the Vision-Transformer-style patchify step in module form.

**Anatomy.**
```python
class Patchify(nn.Module):
    def __init__(self, in_channels, patch, embed_dim):
        super().__init__()                            # nn-module-subclass.
        self.split = Rearrange(                       # rearrange-as-sequential-layer.
            'b c (h p1) (w p2) -> b (h w) (p1 p2 c)',
            p1=patch, p2=patch,
        )
        self.proj = nn.Linear(patch * patch * in_channels, embed_dim)
    def forward(self, x):
        return self.proj(self.split(x))
```

**Why store `Rearrange` as `self.split` and not call `einops.rearrange` inside `forward`.** Three reasons: (a) it's the same parameter-free op in both cases, but the module form shows up in `print(model)` and `model.named_children()`, which makes debugging easier; (b) the pattern string + axis kwargs are validated ONCE at construction time, not on every forward; (c) the module discipline scales — when you later want to `nn.Sequential(Patchify(...), TransformerBlock(...))`, the subclass fits in seamlessly.

### Composite Exercise — Patchify subclass: nn.Module wrapping an einops Rearrange + Linear head

**Atoms exercised together**: `nn-module-subclass`, `rearrange-as-sequential-layer`

Implement `cx3_make_patchify_cls()` — return a `Patchify` class.

Contract:
- `Patchify(in_channels: int, patch: int, embed_dim: int)`.
- Inside `__init__`, call `super().__init__()` first (atom: nn-module-subclass).
- Register exactly two children by attribute assignment:
  - `self.split = Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=patch, p2=patch)` (atom: rearrange-as-sequential-layer).
  - `self.proj = nn.Linear(patch * patch * in_channels, embed_dim)`.
- `forward(self, x)` returns `self.proj(self.split(x))`.

The test checks:
1. Returned value is a class.
2. Instances are `nn.Module`s.
3. `named_children()` is exactly `{'split', 'proj'}`.
4. `self.split` is an `einops.layers.torch.Rearrange`.
5. `self.proj` is `nn.Linear` with the correct in/out features.
6. Forward: `(B, C, H, W)` -> `(B, (H/patch)*(W/patch), embed_dim)` for divisible shapes.
7. The output of `forward(x)` matches the manual reference `self.proj(Rearrange(...)(x))`.
8. `model.parameters()` contains `proj.weight` and `proj.bias` (Rearrange has no learnable params).

In [ ]:
def cx3_make_patchify_cls():
    class Patchify(nn.Module):
        def __init__(self, in_channels, patch, embed_dim):
            # Atom A (nn-module-subclass).
            super().__init__()
            # Atom B (rearrange-as-sequential-layer): Rearrange as a child Module.
            self.split = Rearrange(
                'b c (h p1) (w p2) -> b (h w) (p1 p2 c)',
                p1=patch, p2=patch,
            )
            self.proj = nn.Linear(patch * patch * in_channels, embed_dim)

        def forward(self, x):
            return self.proj(self.split(x))

    return Patchify


<details><summary>Show solution — cx3</summary>

```python
def cx3_make_patchify_cls():
    class Patchify(nn.Module):
        def __init__(self, in_channels, patch, embed_dim):
            # Atom A (nn-module-subclass).
            super().__init__()
            # Atom B (rearrange-as-sequential-layer): Rearrange as a child Module.
            self.split = Rearrange(
                'b c (h p1) (w p2) -> b (h w) (p1 p2 c)',
                p1=patch, p2=patch,
            )
            self.proj = nn.Linear(patch * patch * in_channels, embed_dim)

        def forward(self, x):
            return self.proj(self.split(x))

    return Patchify
```

`Rearrange` is registered as a child purely by attribute assignment — the module machinery in `nn.Module.__setattr__` detects that the RHS is itself a Module. If you instead stored a `functools.partial(einops.rearrange, ...)`, it would NOT appear in `children()` and wouldn't move with `.to(device)` (admittedly Rearrange has nothing to move, but the principle holds for any parameterless layer).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx3',
        'subtopics': ["PyTorch: nn.Module subclassing", "Einops: Rearrange as nn.Sequential layer"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()